In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

print("Libraries imported successfully.")

Libraries imported successfully.


## Mapping Day 1 Raw Python Agent to LangChain

In the Day 1 raw Python agent, I directly created the API client and manually sent requests to the Groq model. In LangChain, the `ChatGroq` class acts as the LLM wrapper and provides a standardized interface for interacting with the model.

In Day 1, tools were ordinary Python functions registered inside an `available_tools` dictionary. In LangChain, tools can be created using the `@tool` decorator or the `Tool` class, allowing LangChain to expose their names, descriptions, and arguments to the model.

In Day 1, I manually implemented the agent loop that checked for function calls, executed tools, returned observations, and repeated the process. LangChain's agent abstractions automate much of this orchestration through components such as agents and `AgentExecutor`.

In Day 1, conversation state and working memory were manually stored in Python structures. LangChain provides memory and message-history abstractions that can manage conversational context more systematically.

Therefore, LangChain does not fundamentally change the agent architecture. Instead, it provides reusable abstractions that reduce the amount of low-level orchestration code that the developer must write.

In [3]:
load_dotenv(override=True)

groq_api_key = os.getenv("XAI_API_KEY")

print("Key loaded:", bool(groq_api_key))

if groq_api_key:
    print("Key prefix:", groq_api_key[:4])
    print("Key length:", len(groq_api_key))
else:
    print("ERROR: XAI_API_KEY not found.")

Key loaded: True
Key prefix: gsk_
Key length: 56


In [4]:
#Create the LangChain LLM
MODEL = "llama-3.3-70b-versatile"
llm = ChatGroq(
    model=MODEL,
    api_key=groq_api_key,
    temperature=0
)
print("LangChain Groq model created successfully.")
print("Model:", MODEL)

LangChain Groq model created successfully.
Model: llama-3.3-70b-versatile


In [7]:
response = llm.invoke(
    "Say hello in one sentence."
)
print(response.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [6]:
print("Response type:", type(response))
print("\nContent:")
print(response.content)

Response type: <class 'langchain_core.messages.ai.AIMessage'>

Content:
Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [7]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    ("human", "{question}")
])
print("Prompt created successfully.")

Prompt created successfully.


In [8]:
formatted_prompt = prompt.invoke({
    "question": "What is machine learning?"
})
print(formatted_prompt)

messages=[SystemMessage(content='You are a helpful AI assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is machine learning?', additional_kwargs={}, response_metadata={})]


In [9]:
chain = prompt | llm
print("LCEL chain created successfully.")

LCEL chain created successfully.


## LCEL Pipe (`|`) Syntax 

In LangChain, both `ChatPromptTemplate` and `ChatGroq` implement the `Runnable` interface, which overloads Python's `__or__` method (normally used for bitwise OR). When you write `prompt | llm`, this is not a bitwise operation — it's LangChain constructing a `RunnableSequence` object that chains the two components together.

At invocation time (`chain.invoke(...)`), the input flows through each `Runnable` in order: the dictionary passed in is first consumed by `prompt`, which formats it into a list of chat messages, and that output is then passed directly as input to `llm`, which sends it to the model and returns the response. Each component in the pipeline only needs to implement a common `invoke()` method, which is what allows arbitrarily different components (prompts, LLMs, output parsers, retrievers, etc.) to be composed with the same `|` syntax.

In [10]:
response = chain.invoke({
    "question": "What is machine learning?"
})
print(response.content)

**Machine Learning: A Definition**

Machine learning is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to learn from data, make decisions, and improve their performance over time. This is achieved without being explicitly programmed for each specific task.

**Key Characteristics:**

1. **Data-driven**: Machine learning relies on large amounts of data to learn patterns, relationships, and trends.
2. **Self-improvement**: Machine learning models can improve their performance over time through experience and learning from data.
3. **Autonomy**: Machine learning models can make decisions and take actions without human intervention.
4. **Adaptability**: Machine learning models can adapt to new data, environments, and tasks.

**Types of Machine Learning:**

1. **Supervised Learning**: The model learns from labeled data to make predictions or classify new data.
2. **Unsupervised Learning**: The model discovers patterns an

In [11]:
response = chain.invoke({
    "question": "Explain why a cricket pitch is 22 yards long."
})
print(response.content)

The length of a cricket pitch is indeed 22 yards (20.12 meters), and this measurement has an interesting history. The origin of the 22-yard length is often attributed to the fact that this was the distance between the bowling crease and the batting crease in the early days of cricket.

In the 18th century, when cricket was first formalized, the game was played on rough, uneven fields with no standardized equipment or rules. The distance between the two sets of stumps (wickets) was not fixed, and it varied from ground to ground.

However, in 1774, the Laws of Cricket were first written down, and they specified that the pitch should be 22 yards long. This measurement was likely chosen because it was the distance between the bowling crease and the batting crease at the Artillery Ground in London, which was a prominent cricket venue at the time.

Another theory suggests that the 22-yard length was chosen because it was equivalent to the length of a chain, which was a common unit of measure

In [12]:
question = "Explain the difference between a chatbot and an AI agent two diffrences in table format with example."
response = chain.invoke({
    "question": question
})
print("Question:")
print(question)
print("\nLangChain Response:")
print(response.content)

Question:
Explain the difference between a chatbot and an AI agent two diffrences in table format with example.

LangChain Response:
Here are two differences between a chatbot and an AI agent in table format with examples:

| **Characteristics** | **Chatbot** | **AI Agent** |
| --- | --- | --- |
| **1. Purpose** | Designed to simulate human-like conversations, primarily for customer support or information retrieval. Example: A chatbot on a website that helps users find answers to frequently asked questions. | Designed to perform specific tasks, make decisions, and take actions autonomously. Example: A virtual assistant like Alexa that can control smart home devices, play music, and set reminders. |
| **2. Intelligence and Autonomy** | Typically rule-based, following pre-defined scripts and workflows. Example: A chatbot that uses pre-defined responses to answer user queries, without the ability to learn or adapt. | Can learn from data, adapt to new situations, and make decisions based o

In [8]:
# Import LangChain's tool decorator for converting Python functions into tools.
from langchain_core.tools import tool
import ast
import operator
print("LangChain tool decorator imported successfully.")

LangChain tool decorator imported successfully.


In [9]:
# Define the allowed mathematical operators for safe evaluation.
_ALLOWED_BINOPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}

_ALLOWED_UNARYOPS = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}

In [10]:
# Recursively evaluate only safe mathematical expressions.
def _evaluate_math_node(node):

    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value

    if isinstance(node, ast.BinOp):
        if type(node.op) not in _ALLOWED_BINOPS:
            raise ValueError("Unsupported operator.")

        left = _evaluate_math_node(node.left)
        right = _evaluate_math_node(node.right)

        return _ALLOWED_BINOPS[type(node.op)](left, right)

    if isinstance(node, ast.UnaryOp):
        if type(node.op) not in _ALLOWED_UNARYOPS:
            raise ValueError("Unsupported unary operator.")

        return _ALLOWED_UNARYOPS[type(node.op)](
            _evaluate_math_node(node.operand)
        )

    raise ValueError("Unsupported expression.")


print("_evaluate_math_node is ready.")

_evaluate_math_node is ready.


In [11]:
# Define the calculator as a LangChain tool for exact arithmetic operations.
@tool
def calculator(expression: str) -> dict:
    """
    Perform an exact mathematical calculation from a valid arithmetic expression.
    
    Use this tool whenever the user requires numerical calculation such as
    addition, subtraction, multiplication, division, or exponentiation.
    """
    try:
        tree = ast.parse(expression, mode="eval")
        result = _evaluate_math_node(tree.body)  
        return {
            "result": result
        }
        
    except Exception as e:
        return {
            "error": str(e)
        }
print("Calculator tool created.")
print("Name:", calculator.name)
print("Description:", calculator.description)

Calculator tool created.
Name: calculator
Description: Perform an exact mathematical calculation from a valid arithmetic expression.

Use this tool whenever the user requires numerical calculation such as
addition, subtraction, multiplication, division, or exponentiation.


In [12]:
# Test the LangChain calculator tool directly before giving it to the agent.

print(calculator.invoke({
    "expression": "125 * 8"
}))

print(calculator.invoke({
    "expression": "100 / 4"
}))

{'result': 1000}
{'result': 25.0}


In [13]:
# Create the same local weather dataset that we used in the Day 1 agent.
WEATHER_DATA = {
    "Lahore": {
        "temperature_c": 38,
        "condition": "Sunny"
    },
    "Islamabad": {
        "temperature_c": 31,
        "condition": "Partly cloudy"
    },
    "Karachi": {
        "temperature_c": 34,
        "condition": "Humid"
    }
}
print("Weather dataset created successfully.")

Weather dataset created successfully.


In [14]:
# Convert the weather function into a LangChain tool with a clear model-facing description.
@tool
def get_weather(city: str) -> dict:
    """
    Get weather information for a specific city.
    
    Use this tool whenever the user asks about the current weather,
    temperature, or weather condition of a city.
    
    Supported cities include Lahore, Islamabad, and Karachi.
    """ 
    city = city.strip()
    
    if city not in WEATHER_DATA:
        return {
            "error": f"Weather data unavailable for {city}."
        }
    
    return {
        "city": city,
        **WEATHER_DATA[city]
    }
print("Weather tool created.")
print("Name:", get_weather.name)
print("Description:", get_weather.description)

Weather tool created.
Name: get_weather
Description: Get weather information for a specific city.

Use this tool whenever the user asks about the current weather,
temperature, or weather condition of a city.

Supported cities include Lahore, Islamabad, and Karachi.


In [20]:
# Test the weather tool with both a supported and unsupported city.
print(get_weather.invoke({
    "city": "Lahore"
}))
print(get_weather.invoke({
    "city": "Faisalabad"
}))

{'city': 'Lahore', 'temperature_c': 38, 'condition': 'Sunny'}
{'error': 'Weather data unavailable for Faisalabad.'}


In [15]:
# Create a small CSV file that will act as our local product database.
import pandas as pd
products_data = {
    "product_id": [101, 102, 103, 104],
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor"],
    "price": [120000, 2500, 4500, 35000],
    "category": ["Electronics", "Accessories", "Accessories", "Electronics"]
}
products_df = pd.DataFrame(products_data)
CSV_PATH = "products.csv"
products_df.to_csv(
    CSV_PATH,
    index=False
)

print(f"CSV created successfully: {CSV_PATH}")

CSV created successfully: products.csv


In [22]:
# Read and display the product CSV to verify that the data source is available.
df = pd.read_csv(CSV_PATH)
print(df)

   product_id   product   price     category
0         101    Laptop  120000  Electronics
1         102     Mouse    2500  Accessories
2         103  Keyboard    4500  Accessories
3         104   Monitor   35000  Electronics


In [16]:
# Create a LangChain tool that searches the real CSV file for a requested product.
@tool
def lookup_product(product_name: str) -> dict:
    """
    Search the local products CSV file for a product and return its
    product ID, name, price, and category.
    
    Use this tool when the user asks about the price, category,
    or details of a product.
    """
    try:
        df = pd.read_csv(CSV_PATH)
        matches = df[
            df["product"].str.lower() == product_name.strip().lower()
        ]
        if matches.empty:
            return {
                "error": f"Product '{product_name}' was not found."
            }
        product = matches.iloc[0]
        return {
            "product_id": int(product["product_id"]),
            "product": product["product"],
            "price": float(product["price"]),
            "category": product["category"]
        }
    except Exception as e:
        return {
            "error": str(e)
        }
print("Product lookup tool created.")
print("Name:", lookup_product.name)
print("Description:", lookup_product.description)

Product lookup tool created.
Name: lookup_product
Description: Search the local products CSV file for a product and return its
product ID, name, price, and category.

Use this tool when the user asks about the price, category,
or details of a product.


In [24]:
# Test the product lookup tool using a product that exists in the CSV file.
result = lookup_product.invoke({
    "product_name": "Laptop"
})
print(result)

{'product_id': 101, 'product': 'Laptop', 'price': 120000.0, 'category': 'Electronics'}


In [25]:
# Test how the tool handles a product that does not exist in the CSV.
result = lookup_product.invoke({
    "product_name": "iPhone"
})
print(result)

{'error': "Product 'iPhone' was not found."}


In [26]:
# Register all three LangChain tools so they can be supplied to an agent.
tools = [
    calculator,
    get_weather,
    lookup_product
]
print("Registered tools:")
for tool_item in tools:
    print("-", tool_item.name)

Registered tools:
- calculator
- get_weather
- lookup_product


In [27]:
# Inspect the descriptions that LangChain exposes to the LLM for tool selection.
for tool_item in tools:
    print("=" * 60)
    print("TOOL:", tool_item.name)
    print("DESCRIPTION:")
    print(tool_item.description)

TOOL: calculator
DESCRIPTION:
Perform an exact mathematical calculation from a valid arithmetic expression.

Use this tool whenever the user requires numerical calculation such as
addition, subtraction, multiplication, division, or exponentiation.
TOOL: get_weather
DESCRIPTION:
Get weather information for a specific city.

Use this tool whenever the user asks about the current weather,
temperature, or weather condition of a city.

Supported cities include Lahore, Islamabad, and Karachi.
TOOL: lookup_product
DESCRIPTION:
Search the local products CSV file for a product and return its
product ID, name, price, and category.

Use this tool when the user asks about the price, category,
or details of a product.


In [28]:
# Inspect the input schemas LangChain generated from the Python function parameters.
for tool_item in tools:
    print("=" * 60)
    print("TOOL:", tool_item.name)
    print("SCHEMA:")
    print(tool_item.args_schema.model_json_schema())

TOOL: calculator
SCHEMA:
{'description': 'Perform an exact mathematical calculation from a valid arithmetic expression.\n\nUse this tool whenever the user requires numerical calculation such as\naddition, subtraction, multiplication, division, or exponentiation.', 'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculator', 'type': 'object'}
TOOL: get_weather
SCHEMA:
{'description': 'Get weather information for a specific city.\n\nUse this tool whenever the user asks about the current weather,\ntemperature, or weather condition of a city.\n\nSupported cities include Lahore, Islamabad, and Karachi.', 'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weather', 'type': 'object'}
TOOL: lookup_product
SCHEMA:
{'description': 'Search the local products CSV file for a product and return its\nproduct ID, name, price, and category.\n\nUse this tool when the user asks about the price, ca

## How Tool Docstrings Work as Part of the Prompt

When we use `@tool`, LangChain doesn't just keep the docstring as a comment — it sends it straight to the LLM as the tool's description. So the model never sees our code, it only sees the docstring. That's how it decides which tool to pick and when.

This is why we wrote docstrings that say things like "use this tool when the user asks about weather" instead of just "gets weather data." The model isn't reading it like a person would — it's using it as an instruction to decide, so the more clearly we describe *when* to use the tool, the better the model picks the right one.

In [29]:
# Bind the registered tools to the Groq model so it can request tool calls.
llm_with_tools = llm.bind_tools(tools)
print("Tools successfully bound to the LangChain LLM.")

Tools successfully bound to the LangChain LLM.


In [30]:
# Ask the model a calculation question and inspect whether it requests the calculator tool.
response = llm_with_tools.invoke(
    "What is 25 multiplied by 40?"
)
print(response)
# Display any tool calls requested by the model.
print("Tool calls:")
print(response.tool_calls)

content='' additional_kwargs={'tool_calls': [{'id': 'yna8z0j30', 'function': {'arguments': '{"expression":"25 * 40"}', 'name': 'calculator'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 524, 'total_tokens': 541, 'completion_time': 0.042725205, 'completion_tokens_details': None, 'prompt_time': 0.030967534, 'prompt_tokens_details': None, 'queue_time': 0.199369426, 'total_time': 0.073692739}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--019fefa9-9005-74a2-bf56-ef0aea7a3b12-0' tool_calls=[{'name': 'calculator', 'args': {'expression': '25 * 40'}, 'id': 'yna8z0j30', 'type': 'tool_call'}] usage_metadata={'input_tokens': 524, 'output_tokens': 17, 'total_tokens': 541}
Tool calls:
[{'name': 'calculator', 'args': {'expression': '25 * 40'}, 'id': 'yna8z0j30', 'type': 'tool_call'}]


In [31]:
# Ask the model a weather question and inspect its requested tool call.
response = llm_with_tools.invoke(
    "What is the weather in Lahore?"
)
print("Tool calls:")
print(response.tool_calls)

Tool calls:
[{'name': 'get_weather', 'args': {'city': 'Lahore'}, 'id': '3a3bnz0bx', 'type': 'tool_call'}]


In [32]:
# Ask the model for product information to verify that it can select the CSV-backed tool.
response = llm_with_tools.invoke(
    "What is the price of the Laptop?"
)
print("Tool calls:")
print(response.tool_calls)

Tool calls:
[{'name': 'lookup_product', 'args': {'product_name': 'Laptop'}, 'id': 't3aerarq2', 'type': 'tool_call'}]


In [33]:
# compact summary confirming that all Task 2 tools are ready for the agent.
print("=" * 60)
print("TASK 2 — TOOL REGISTRATION COMPLETE")
print("=" * 60)
for tool_item in tools:
    print(f"Tool: {tool_item.name}")
    print(f"Description: {tool_item.description}")
    print()

TASK 2 — TOOL REGISTRATION COMPLETE
Tool: calculator
Description: Perform an exact mathematical calculation from a valid arithmetic expression.

Use this tool whenever the user requires numerical calculation such as
addition, subtraction, multiplication, division, or exponentiation.

Tool: get_weather
Description: Get weather information for a specific city.

Use this tool whenever the user asks about the current weather,
temperature, or weather condition of a city.

Supported cities include Lahore, Islamabad, and Karachi.

Tool: lookup_product
Description: Search the local products CSV file for a product and return its
product ID, name, price, and category.

Use this tool when the user asks about the price, category,
or details of a product.



In [2]:
# Import Groq, LangChain agents, prompts, and the executor required for the agent.
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

## Note on Agent Construction API

This notebook uses `create_agent` (LangChain 1.x, LangGraph-based) instead of the `create_tool_calling_agent` + `AgentExecutor` combo named in the task. Attempting to use `AgentExecutor` in this environment raised a Python/dependency version conflict, so `create_agent` was used instead — it handles the same underlying reasoning loop (model → tool call → tool execution → observation → repeat) via LangGraph internally, and is also the API LangChain now recommends going forward since `AgentExecutor` is being deprecated. `.stream(stream_mode="updates")` was used in place of `verbose=True` to capture the step-by-step trace, since `create_agent` does not expose a `verbose` flag.

In [4]:
# Load the Groq API key from the .env file and verify that it is available.
load_dotenv(override=True)

GROQ_API_KEY = os.getenv("XAI_API_KEY")

print("Groq API key loaded:", bool(GROQ_API_KEY))

Groq API key loaded: True


In [5]:
# Create the LangChain ChatGroq model that will make tool-calling decisions.
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)
print("LangChain Groq model created successfully.")

LangChain Groq model created successfully.


In [17]:
# Collect the LangChain tools that will be available to the agent.
agent_tools = [
    calculator,
    get_weather,
    lookup_product
]

print("Tools registered:")
for tool in agent_tools:
    print("-", tool.name)

Tools registered:
- calculator
- get_weather
- lookup_product


In [18]:
# Create a LangChain 1.x agent that can automatically select and execute our tools.
from langchain.agents import create_agent
agent = create_agent(
    model=llm,
    tools=agent_tools,
    system_prompt=(
        "You are a helpful AI agent. "
        "Use the available tools whenever they are needed. "
        "For calculations, use the calculator tool. "
        "For weather questions, use the get_weather tool. "
        "For product information, use the lookup_product tool. "
        "After receiving tool results, provide a clear final answer."
    )
)

print("LangChain agent created successfully.")
print("Agent type:", type(agent))

LangChain agent created successfully.
Agent type: <class 'langgraph.graph.state.CompiledStateGraph'>


In [19]:
# Run the agent on a simple calculation task and display its final response.
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 125 * 8?"
        }
    ]
})
print("Agent completed.")
print("\nFinal response:")
print(result["messages"][-1].content)

Agent completed.

Final response:
The result of 125 * 8 is 1000.


In [20]:
# Inspect every message produced during the agent execution to understand the tool-calling flow.
for i, message in enumerate(result["messages"]):
    print("=" * 70)
    print(f"Message {i}")
    print("Type:", type(message).__name__)
    print("Content:", message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:", message.tool_calls)

Message 0
Type: HumanMessage
Content: What is 125 * 8?
Message 1
Type: AIMessage
Content: 
Tool calls: [{'name': 'calculator', 'args': {'expression': '125 * 8'}, 'id': 'v09tfvs8x', 'type': 'tool_call'}]
Message 2
Type: ToolMessage
Content: {"result": 1000}
Message 3
Type: AIMessage
Content: The result of 125 * 8 is 1000.


In [21]:
# Test whether the agent can select and execute the weather tool automatically.
weather_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the weather in Lahore?"
        }
    ]
})
print("Final response:")
print(weather_result["messages"][-1].content)

Final response:
The current weather in Lahore is sunny with a temperature of 38 degrees Celsius.


In [ ]:
# Test whether the agent can select the CSV-backed product lookup tool.
product_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of the Laptop?"
        }
    ]
})
print("Final response:")
print(product_result["messages"][-1].content)

Final response:
The price of the Laptop is 120000.0.


In [23]:
# Run a multi-step question that requires the agent to use multiple tools.
multi_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "What is the weather in Lahore and Islamabad, "
                "and tell me which city is warmer."
            )
        }
    ]
})
print("Final response:")
print(multi_result["messages"][-1].content)

Final response:
The weather in Lahore is sunny with a temperature of 38 degrees Celsius. The weather in Islamabad is partly cloudy with a temperature of 31 degrees Celsius. Lahore is warmer than Islamabad.


In [24]:
# Stream the agent execution so we can observe model, tool, and result events as they occur.
trace_events = []
for event in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "What is the weather in Lahore and Islamabad, "
                    "and tell me which city is warmer."
                )
            }
        ]
    },
    stream_mode="updates"
):
    trace_events.append(event)
    print("=" * 70)
    print(event)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8vphjp66h', 'function': {'arguments': '{"city":"Lahore"}', 'name': 'get_weather'}, 'type': 'function'}, {'id': 't2y8kwwgt', 'function': {'arguments': '{"city":"Islamabad"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 587, 'total_tokens': 617, 'completion_time': 0.063358404, 'completion_tokens_details': None, 'prompt_time': 0.101116556, 'prompt_tokens_details': None, 'queue_time': 0.008942066, 'total_time': 0.16447496}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fefc8-bfe9-7bb0-a897-dbfdca2f694a-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Lahore'}, 'id': '8vphjp66h', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': 'Islamabad'}, 'id': 't2y

In [25]:
# Convert the streamed events into a simpler trace showing model actions and tool observations.
for event in trace_events:
    for node_name, node_data in event.items():

        print("=" * 70)
        print("NODE:", node_name)

        if "messages" not in node_data:
            continue

        for message in node_data["messages"]:

            message_type = type(message).__name__

            print("MESSAGE TYPE:", message_type)

            if hasattr(message, "tool_calls") and message.tool_calls:
                print("ACTION:")
                for call in message.tool_calls:
                    print("  Tool:", call["name"])
                    print("  Arguments:", call["args"])

            elif message_type == "ToolMessage":
                print("OBSERVATION:")
                print(" ", message.content)

            elif message.content:
                print("CONTENT:")
                print(" ", message.content)

NODE: model
MESSAGE TYPE: AIMessage
ACTION:
  Tool: get_weather
  Arguments: {'city': 'Lahore'}
  Tool: get_weather
  Arguments: {'city': 'Islamabad'}
NODE: tools
MESSAGE TYPE: ToolMessage
OBSERVATION:
  {"city": "Lahore", "temperature_c": 38, "condition": "Sunny"}
NODE: tools
MESSAGE TYPE: ToolMessage
OBSERVATION:
  {"city": "Islamabad", "temperature_c": 31, "condition": "Partly cloudy"}
NODE: model
MESSAGE TYPE: AIMessage
CONTENT:
  The weather in Lahore is sunny with a temperature of 38 degrees Celsius. The weather in Islamabad is partly cloudy with a temperature of 31 degrees Celsius. Lahore is warmer than Islamabad.


In [26]:
# Label the observable execution stages so they can be compared directly with the Day 1 ReAct loop.

print("=" * 70)
print("ANNOTATED AGENT TRACE")
print("=" * 70)

for event in trace_events:

    for node_name, node_data in event.items():

        if "messages" not in node_data:
            continue

        for message in node_data["messages"]:

            message_type = type(message).__name__

            if message_type == "AIMessage" and getattr(message, "tool_calls", None):
                print("\n[ACT]")
                for call in message.tool_calls:
                    print(
                        f"Agent selected tool '{call['name']}' "
                        f"with arguments {call['args']}"
                    )

            elif message_type == "ToolMessage":
                print("\n[OBSERVE]")
                print("Tool returned:", message.content)

            elif message_type == "AIMessage" and message.content:
                print("\n[FINAL / MODEL OUTPUT]")
                print(message.content)

ANNOTATED AGENT TRACE

[ACT]
Agent selected tool 'get_weather' with arguments {'city': 'Lahore'}
Agent selected tool 'get_weather' with arguments {'city': 'Islamabad'}

[OBSERVE]
Tool returned: {"city": "Lahore", "temperature_c": 38, "condition": "Sunny"}

[OBSERVE]
Tool returned: {"city": "Islamabad", "temperature_c": 31, "condition": "Partly cloudy"}

[FINAL / MODEL OUTPUT]
The weather in Lahore is sunny with a temperature of 38 degrees Celsius. The weather in Islamabad is partly cloudy with a temperature of 31 degrees Celsius. Lahore is warmer than Islamabad.


In [27]:
# Compare the manual Day 1 agent architecture with the LangChain implementation.

comparison = """
DAY 1 — RAW PYTHON AGENT

The application manually created the model request, detected function calls,
looked up the requested Python function, executed the tool, constructed the
tool result, updated conversation state, and repeated the loop.

DAY 2 — LANGCHAIN AGENT

LangChain provides the agent orchestration layer. The developer mainly defines
the model and tools, while LangChain manages tool routing, message state,
tool execution, and repeated model/tool interactions.

SIMILARITY

Both systems follow the same fundamental architecture:
User → Model → Tool Call → Tool Execution → Observation → Model → Final Answer.

DIFFERENCE

In Day 1, the orchestration logic was explicitly visible in our Python code.
In LangChain, much of that orchestration is implemented inside LangChain's
agent runtime, so the code becomes shorter but some internal behavior becomes
less explicit.
"""

print(comparison)


DAY 1 — RAW PYTHON AGENT

The application manually created the model request, detected function calls,
looked up the requested Python function, executed the tool, constructed the
tool result, updated conversation state, and repeated the loop.

DAY 2 — LANGCHAIN AGENT

LangChain provides the agent orchestration layer. The developer mainly defines
the model and tools, while LangChain manages tool routing, message state,
tool execution, and repeated model/tool interactions.

SIMILARITY

Both systems follow the same fundamental architecture:
User → Model → Tool Call → Tool Execution → Observation → Model → Final Answer.

DIFFERENCE

In Day 1, the orchestration logic was explicitly visible in our Python code.
In LangChain, much of that orchestration is implemented inside LangChain's
agent runtime, so the code becomes shorter but some internal behavior becomes
less explicit.



In [28]:
#task 4
# Import LangChain message-history components for maintaining conversation context.
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

print("Memory components imported successfully.")

Memory components imported successfully.


In [29]:
# Create an in-memory dictionary that stores conversation history for each session.
store = {}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

print("Conversation history store created.")

Conversation history store created.


In [35]:
# Create a separate message-history store for each conversation session.
from langchain_core.messages import HumanMessage, AIMessage
conversation_store = {}
def get_history(session_id):
    if session_id not in conversation_store:
        conversation_store[session_id] = []
    return conversation_store[session_id]

print("Conversation memory store created.")

Conversation memory store created.


In [36]:
# Create a helper function that sends previous conversation messages to the agent.

def run_agent_with_memory(user_message, session_id):

    history = get_history(session_id)

    messages = history + [
        HumanMessage(content=user_message)
    ]

    result = agent.invoke({
        "messages": messages
    })

    final_message = result["messages"][-1]

    history.append(HumanMessage(content=user_message))
    history.append(AIMessage(content=final_message.content))

    return final_message.content

print("Memory-aware agent function created.")

Memory-aware agent function created.


In [42]:
agent_with_memory = RunnableWithMessageHistory(
    agent,
    get_session_history,
    input_messages_key="messages",
    history_messages_key="history"
)

d:\Netixsol_Intern_Projects\.venv-1\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [47]:
# Turn 1: Ask for the Laptop price and save the conversation in memory.

response1 = run_agent_with_memory(
    "What is the price of the Laptop?",
    "budget_client"
)

print("Turn 1:")
print(response1)

Turn 1:
The price of the Laptop is 120000.0.


In [48]:
# Turn 2: Refer to the Laptop indirectly and ask for a comparison with the Monitor.

response2 = run_agent_with_memory(
    "Now compare it to the Monitor.",
    "budget_client"
)

print("Turn 2:")
print(response2)

Turn 2:
The Laptop is priced at 120000.0, while the Monitor is priced at 35000.0. The Laptop is more expensive than the Monitor.


In [49]:
# Turn 3: Use the previous product discussion to make a budget-conscious recommendation.

response3 = run_agent_with_memory(
    "Which one should I recommend to a budget-conscious client?",
    "budget_client"
)

print("Turn 3:")
print(response3)

Turn 3:
Based on the prices, I would recommend the Monitor to a budget-conscious client, as it is significantly cheaper than the Laptop. The Monitor costs 35000.0, while the Laptop costs 120000.0.


In [50]:
# Display the stored human and AI messages to verify that conversation memory is working.

history = get_history("budget_client")

print("=" * 70)
print("CONVERSATION HISTORY")
print("=" * 70)

for i, message in enumerate(history, start=1):
    print(f"\nMessage {i}")
    print("Type:", type(message).__name__)
    print("Content:", message.content)

CONVERSATION HISTORY

Message 1
Type: HumanMessage
Content: What is the price of the Laptop?

Message 2
Type: AIMessage
Content: The price of the Laptop is 120000.0.

Message 3
Type: HumanMessage
Content: Now compare it to the Monitor.

Message 4
Type: AIMessage
Content: The Laptop is priced at 120000.0, while the Monitor is priced at 35000.0. The Laptop is more expensive than the Monitor.

Message 5
Type: HumanMessage
Content: Which one should I recommend to a budget-conscious client?

Message 6
Type: AIMessage
Content: Based on the prices, I would recommend the Monitor to a budget-conscious client, as it is significantly cheaper than the Laptop. The Monitor costs 35000.0, while the Laptop costs 120000.0.

Message 7
Type: HumanMessage
Content: What is the price of the Laptop?

Message 8
Type: AIMessage
Content: The price of the Laptop is 120000.0.

Message 9
Type: HumanMessage
Content: Now compare it to the Monitor.

Message 10
Type: AIMessage
Content: The Laptop is priced at 120000.0

## Task 4 — Conversation Memory

`RunnableWithMessageHistory` was tried first (`agent_with_memory`), but it's built for chains with separate "new input" and "history" slots — our `create_agent` agent expects one unified `messages` list instead, so this caused malformed input and generic fallback replies.

The working version uses a manual `run_agent_with_memory()` helper with a session dict (`conversation_store`), which builds the full message list (history + new turn) before each `agent.invoke()` call — same idea as Day 1's manual memory, just wrapped for reuse.

Turn 1 sets the Laptop price, turn 2 refers to it as "it," and turn 3 needs both products remembered to make a recommendation — showing the agent is actually using prior context, not just storing it.

In [60]:
# Task 5 — Structured Output & Error Handling
# Define a Pydantic schema that forces the agent's final answer into a fixed structure.
from pydantic import BaseModel, Field

class ProductRecommendation(BaseModel):
    """Structured recommendation the agent must return as its final answer."""
    recommended_product: str = Field(description="Name of the recommended product")
    price: float = Field(description="Price of the recommended product")
    reason: str = Field(description="Short reason for the recommendation")
    budget_conscious: bool = Field(description="Whether this pick suits a budget-conscious client")

print("Structured output schema defined.")

Structured output schema defined.


In [61]:
# Create a new agent instance that forces its final answer into the ProductRecommendation schema.
structured_agent = create_agent(
    model=llm,
    tools=agent_tools,
    system_prompt=(
        "You are a helpful AI agent. Use tools when needed. "
        "Once you have enough information, provide a final structured recommendation."
    ),
    response_format=ProductRecommendation
)

print("Structured-output agent created.")

Structured-output agent created.


In [62]:
# Run the structured agent and inspect both the raw message and the validated structured object.
result = structured_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Compare the Laptop and Monitor, and recommend one for a budget-conscious client."
        }
    ]
})

print("Raw final message:")
print(result["messages"][-1].content)

print("\nStructured response:")
print(result["structured_response"])
print("Type:", type(result["structured_response"]))

Raw final message:
{"result": 500.0}

Structured response:
recommended_product='Laptop' price=500.0 reason='The product is affordable and has the necessary features for a budget-conscious client.' budget_conscious=True
Type: <class '__main__.ProductRecommendation'>


In [67]:
from langchain_core.tools import tool
# Create a tool that randomly fails, simulating an unreliable external API.
# Errors are caught INSIDE the tool and returned as data, matching the
# error-handling pattern already used by calculator and lookup_product.
import random

@tool
def get_exchange_rate(currency: str) -> dict:
    """
    Get the exchange rate of a currency against PKR.

    Use this tool when the user asks to convert a price into another currency.
    This tool randomly fails to simulate an unreliable external service.
    """
    try:
        if random.random() < 0.5:
            raise ValueError(f"Exchange rate service temporarily unavailable for {currency}.")

        rates = {"USD": 278.0, "EUR": 300.0, "GBP": 350.0}

        if currency.upper() not in rates:
            return {"error": f"No exchange rate found for {currency}."}

        return {"currency": currency.upper(), "rate_to_pkr": rates[currency.upper()]}

    except Exception as e:
        return {"error": str(e)}

print("Flaky exchange-rate tool created.")

Flaky exchange-rate tool created.


In [69]:
# Register the flaky tool alongside the existing tools and build an agent that knows how to respond to failures.
agent_tools_v2 = [calculator, get_weather, lookup_product, get_exchange_rate]

error_handling_agent = create_agent(
    model=llm,
    tools=agent_tools_v2,
    system_prompt=(
        "You are a helpful AI agent. If a tool fails or returns an error, "
        "briefly explain that the service is temporarily unavailable, "
        "and still help with whatever information you do have."
    )
)

print("Agent with flaky tool created.")

Agent with flaky tool created.


In [70]:
# Run the same request several times to trigger both the success and failure paths.
for attempt in range(1, 4):
    print("=" * 70)
    print(f"Attempt {attempt}")
    result = error_handling_agent.invoke({
        "messages": [
            {"role": "user", "content": "Convert the Laptop price to USD."}
        ]
    })
    print(result["messages"][-1].content)

Attempt 1
The Laptop price in USD is approximately $431.65.
Attempt 2
The Laptop price in USD is approximately $431.65.
Attempt 3
I'm sorry, but the exchange rate service is temporarily unavailable. However, I can tell you that the price of the Laptop is 120000 PKR.


In [71]:
# Inspect the raw message trace to see exactly how a tool failure was represented and recovered from.
for message in result["messages"]:
    if type(message).__name__ == "ToolMessage":
        print("Tool message status:", getattr(message, "status", None))
        print("Tool message content:", message.content)

Tool message status: success
Tool message content: {"error": "Exchange rate service temporarily unavailable for USD."}
Tool message status: success
Tool message content: {"product_id": 101, "product": "Laptop", "price": 120000.0, "category": "Electronics"}


#### Configuring graceful recovery

At first the tool just `raise`d on failure, expecting LangGraph to catch it automatically — it didn't, and `.invoke()` crashed instead. The fix was wrapping the tool's own logic in `try/except` and returning `{"error": ...}` instead of raising, same as `calculator` and `lookup_product` already do. So graceful recovery isn't automatic here — we had to build it ourselves at the tool level, just like Day 1.

#### What LangChain Made Easier vs. Day 1 — and Where the Magic Is

Structured output was much easier — `response_format=ProductRecommendation` handled prompting and validation for us, instead of manually parsing and checking JSON like Day 1. Error handling, though, wasn't automatic like we expected: we assumed `create_agent` would catch tool exceptions on its own, but it didn't, and the run crashed until we added `try/except` ourselves inside the tool. That's the "magic" trap here — LangChain hides a lot, but it also quietly assumes tools handle their own errors, and you only find that out once something breaks.